In [5]:
from transformers import MarianMTModel, MarianTokenizer
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset, concatenate_datasets
import pandas as pd
import torch
import sacrebleu
from tqdm import tqdm

In [6]:
# Load translation model and tokenizer
def load_translation_model(model_name):
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)
    return tokenizer, model

In [7]:
# Translate a sentence
def translate(texts, tokenizer, model):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in outputs]

In [8]:
# Load models
cy2en_tokenizer, cy2en_model = load_translation_model("Helsinki-NLP/opus-mt-cy-en")
en2cy_tokenizer, en2cy_model = load_translation_model("Helsinki-NLP/opus-mt-en-cy")
en2fr_tokenizer, en2fr_model = load_translation_model("Helsinki-NLP/opus-mt-en-fr")
fr2en_tokenizer, fr2en_model = load_translation_model("Helsinki-NLP/opus-mt-fr-en")

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\transformers\models\marian\tokenization_marian.py:177: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\c24082331\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-fr. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either 

In [9]:
# Load SentenceTransformer model
sim_model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2')

In [10]:
sim_model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: DistilBertModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 512, 'bias': True, 'activation_function': 'torch.nn.modules.activation.Tanh'})
)

In [11]:
#Semantic similarity
def semantic_similarity(s1, s2):
    emb1 = sim_model.encode(s1, convert_to_tensor=True)
    emb2 = sim_model.encode(s2, convert_to_tensor=True)
    return util.cos_sim(emb1, emb2).item()

In [12]:
def evaluate(original, back_translated):
    bleu = sacrebleu.corpus_bleu([back_translated], [[original]]).score
    chrf = sacrebleu.corpus_chrf([back_translated], [[original]]).score
    cosine = semantic_similarity(original, back_translated)
    return round(bleu, 2), round(chrf, 2), round(cosine, 4)

In [13]:
# Load Welsh CEFR data
welsh_data = load_dataset("UniversalCEFR/learn_welsh_cy")["train"]

# Filter A1 and A2 level texts
welsh_a1_a2 = welsh_data.filter(lambda example: example["cefr_level"] in ["A1", "A2"])

df = welsh_a1_a2.to_pandas()[["text", "cefr_level"]].dropna().reset_index(drop=True)

In [14]:
welsh_data

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
    num_rows: 1372
})

In [15]:
df

,text,cefr_level
0,"A: Helô, Eryl dw i. Pwy dych chi?\nB: Bore da,...",A1
1,"A: O na, yr heddlu! (Stopio'r car)\nB: Hello, ...",A1
2,"A: Bore da. Sut dych chi?\nB: Iawn, ond wedi b...",A1
3,"Ceri: Noswaith dda, Eryl. Sut wyt ti?\nEryl: D...",A1
4,A: Bore da.\nB: Hmff.\nA: Sut dych chi heddiw?...,A1
...,...,...
1367,Allech chi gyrraedd yn gynnar?,A2
1368,Allet ti gyrraedd yn gynnar?,A2
1369,Allai hi gyrraedd yn gynnar?,A2
1370,Allen nhw gyrraedd yn gynnar?,A2


In [16]:
# Count A1 and A2
df["cefr_level"].value_counts()

cefr_level
A1    764
A2    608
Name: count, dtype: int64

In [18]:
records = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Back-translating"):
    original_cy = row["text"]
    cefr = row["cefr_level"]
    
    try:
        # Welsh to English
        english = translate([original_cy], cy2en_tokenizer, cy2en_model)[0]

        # English to French
        french = translate([english], en2fr_tokenizer, en2fr_model)[0]

        # French back to English
        back_english = translate([french], fr2en_tokenizer, fr2en_model)[0]

        # English back to Welsh
        back_cy = translate([back_english], en2cy_tokenizer, en2cy_model)[0]

        # Evaluate
        bleu, chrf, cosine = evaluate(original_cy, back_cy)

        records.append({"original_welsh": original_cy,
            "translated_english": english,
            "translated_french": french,
            "back_translated_english": back_english,
            "back_translated_welsh": back_cy,
            "cefr_level": cefr,
            "BLEU": bleu,
            "chrF": chrf,
            "cosine_similarity": cosine
        })

    except Exception as e:
        print(f"Error on: {original_cy}\n{e}")

Back-translating: 100%|██████████| 1372/1372 [2:34:07<00:00,  6.74s/it]  


In [19]:
df_bt = pd.DataFrame(records)
df_bt.to_csv("files/welsh_back_translation_french.csv", index=False)

In [20]:
df_bt

,original_welsh,translated_english,translated_french,back_translated_english,back_translated_welsh,cefr_level,BLEU,chrF,cosine_similarity
0,"A: Helô, Eryl dw i. Pwy dych chi?\nB: Bore da,...","And: Very, To: Where do you do?","Et: Très, à: Où faites-vous?","And: Very, to: Where are you doing?","A: Iawn, beth wyt ti'n ei wneud?",A1,0.29,6.99,0.5216
1,"A: O na, yr heddlu! (Stopio'r car)\nB: Hello, ...","And: O, the police!","Et : O, la police !","And: O, the police!","Ac: O, yr heddlu!",A1,0.00,3.23,0.3973
2,"A: Bore da. Sut dych chi?\nB: Iawn, ond wedi b...","And: Good morning, How do you know: That's tir...","Et: Bonjour, comment savez-vous: C'est fatigué...","And: Hello, how do you know: It's tired, but t...","Ac: Helo, sut wyt ti'n gwybod: Mae'n blino, on...",A1,0.46,15.45,0.7102
3,"Ceri: Noswaith dda, Eryl. Sut wyt ti?\nEryl: D...","Well: Good night, though, wins. How do you fee...","Eh bien: Bonne nuit, cependant, gagne. Comment...","Well: Good night, though, wins. How do you fee...","Ond, nos da: Ond, sut rwyt ti'n teimlo?",A1,0.07,6.39,0.3846
4,A: Bore da.\nB: Hmff.\nA: Sut dych chi heddiw?...,And morning: Good morning. B. Where: How do yo...,Et le matin: Bonjour. B. Où: Comment savez-vou...,And in the morning: Hello. B. Where: How do yo...,Ac yn y bore: Helo. Ble ydych chi'n gwybod hed...,A1,0.00,5.77,0.6013
...,...,...,...,...,...,...,...,...,...
1367,Allech chi gyrraedd yn gynnar?,Can you get early?,Tu peux être en avance ?,Can you be early?,A elli di fod yn gynnar?,A2,24.45,38.31,0.4836
1368,Allet ti gyrraedd yn gynnar?,Can you reach early?,Pouvez-vous atteindre tôt ?,Can you reach early?,A elli di gyrraedd gynnar?,A2,17.97,54.72,0.7349
1369,Allai hi gyrraedd yn gynnar?,Could she get early?,Elle pourrait être en avance ?,Could she be early?,A allai hi fod yn gynnar?,A2,26.27,53.57,0.8717
1370,Allen nhw gyrraedd yn gynnar?,Allend them early?,Tu les as prévenus tôt ?,You warned them early?,Rydych yn eu rhybuddio nhw yn gynnar?,A2,22.09,36.11,0.5820
